# KidneyMTL on Google Colab: train, evaluate and download results

<a href="https://colab.research.google.com/github/utkarshhzz/Kidney_research/blob/main/notebooks/KidneyMTL_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:** `Runtime → Change runtime type → GPU` (T4 works; L4/A100 is faster).

**How this notebook survives disconnects:** every finished run (metrics, predictions, plots, logs) is saved to
**Google Drive** (`MyDrive/Kidney_research/results`) and marked `DONE`. When Colab disconnects, open the notebook again,
run sections 1 and 2 again, and finish with the training cell. Finished runs are skipped. Only the run that was in
progress restarts from its beginning. Checkpoints are written to fast local disk instead of Drive, so they don't fill
your Drive quota.

| Section | Run when |
|---|---|
| 1. Setup | every session |
| 2. Data | every session (first time downloads from Kaggle and caches the zips on Drive; later sessions only unzip) |
| 3. Check | first session (and after code updates) |
| 4. Train | until the plan is finished, as many sessions as needed |
| 5. Analyse | after training |
| 6. Download | after analysing (or at any time for partial results) |

## 1. Setup

In [1]:
#@title 1.1 Settings
DRIVE_DIR = "/content/drive/MyDrive/Kidney_research"  #@param {type:"string"}
REPO_URL = "https://github.com/utkarshhzz/Kidney_research.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os
RESULTS_DIR = f"{DRIVE_DIR}/results"      # everything that matters is saved here (Google Drive)
RUNS_DIR = f"{RESULTS_DIR}/runs"
REPO_DIR = "/content/Kidney_research"     # code + data on fast local disk
WORK_DIR = "/content/work"                # per-epoch checkpoints (local, not Drive)
NUM_WORKERS = max(2, min(os.cpu_count() or 2, 8))
print("CPU cores:", os.cpu_count(), "-> data loader workers:", NUM_WORKERS)

CPU cores: 2 -> data loader workers: 2


In [2]:
#@title 1.2 GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU, then run again."
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

name, memory.total [MiB]
Tesla T4, 15360 MiB
torch 2.11.0+cu130 | GPU: Tesla T4


In [3]:
#@title 1.3 Mount Google Drive and get the code
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS_DIR, exist_ok=True)

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

Mounted at /content/drive
Cloning into '/content/Kidney_research'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 82 (delta 15), reused 81 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 1.98 MiB | 32.18 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Kidney_research
9828d63 (HEAD -> main, origin/main, origin/HEAD) Train on Google Colab: notebook, Drive-safe checkpoints, result download/import


In [4]:
#@title 1.4 Install dependencies (torch is already installed on Colab)
!pip install -q -r requirements-colab.txt
import timm, albumentations, torchvision, cv2, sklearn
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__,
      "| albumentations", albumentations.__version__, "| opencv", cv2.__version__, "| sklearn", sklearn.__version__)

torch 2.11.0+cu130 | torchvision 0.26.0+cu130 | timm 1.0.15 | albumentations 2.0.8 | opencv 5.0.0 | sklearn 1.6.1


## 2. Data
Kaggle credentials are taken from (in this order):
1. **Colab Secrets** (key icon in the left sidebar): `KAGGLE_USERNAME` and `KAGGLE_KEY`, with notebook access switched on. This is the recommended option.
2. `kaggle.json` in your Drive folder (`MyDrive/Kidney_research/kaggle.json`)
3. an upload prompt.

⚠️ Use a **new** Kaggle API token. The old one was exposed in an earlier notebook, so revoke it.

In [ ]:
#@title 2.1 Kaggle credentials
import json
from pathlib import Path

creds = {}
try:
    from google.colab import userdata
    creds = {"username": userdata.get("KAGGLE_USERNAME"), "key": userdata.get("KAGGLE_KEY")}
except Exception as e:
    print("Colab secrets not available:", type(e).__name__)
if not all(creds.values()) and (Path(DRIVE_DIR) / "kaggle.json").exists():
    creds = json.loads((Path(DRIVE_DIR) / "kaggle.json").read_text())
if not creds or not all(creds.values()):
    from google.colab import files
    print("Upload your kaggle.json (kaggle.com -> Settings -> API -> Create New Token)")
    uploaded = files.upload()
    creds = json.loads(next(iter(uploaded.values())))

os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]
kdir = Path.home() / ".kaggle"
kdir.mkdir(exist_ok=True)
(kdir / "kaggle.json").write_text(json.dumps({"username": creds["username"], "key": creds["key"]}))
os.chmod(kdir / "kaggle.json", 0o600)
print("Kaggle credentials ready for user:", creds["username"])

Colab secrets not available: SecretNotFoundError
Upload your kaggle.json (kaggle.com -> Settings -> API -> Create New Token)


In [ ]:
#@title 2.2 Download (first time) or unzip from the Drive cache, then verify
!python scripts/download_data.py --cache-dir "{DRIVE_DIR}/datasets"

## 3. Check that everything works (about 3 minutes)
Runs the correctness tests (leak-free folds, detection encode/decode, metrics) and a tiny end-to-end training run.
Do this in the first session and whenever the code changes. If it fails, **don't start training**; copy the error and fix it first.

In [ ]:
#@title 3.1 Tests + quick smoke test
!python tests/test_core.py
!python scripts/smoke_test.py --quick

## 4. Train
Choose a plan and run 4.1 (shows what will run), then 4.2. Progress is stored on Drive, so you can stop at any time and
run 4.2 again later. **Finished runs are skipped.**

| Plan | What | Runs |
|---|---|---|
| `main_fold0_only` | proposed model, 1 fold: **do this first** to measure time per run | 1 |
| `tier1_3folds` | minimum publishable set, 3 folds | 30 |
| `tier1` | minimum publishable set, all folds | 44 |
| `baselines` / `ablations` / `leakage` / `detection` | one group | 35 / 36 / 24 / 1 |
| `all` | complete paper | 101 |

**Time:** look at `minutes` in 4.3 after the first run. On a T4, expect very roughly 1 hour per run for the main
model and less with early stopping. Free Colab gives a few GPU hours a day, so with the free tier start with
`main_fold0_only` and then `tier1_3folds`. Colab Pro (background execution) makes `tier1` or `all` realistic.

**Out of memory?** The runner automatically retries a failed run with half the batch size. You can also lower the
batch sizes below.

In [ ]:
#@title 4.1 Choose a plan (dry run: nothing is trained)
PLAN = "main_fold0_only"  #@param ["main_fold0_only", "tier1_3folds", "tier1", "baselines", "ablations", "leakage", "detection", "all"]
HOURS = 11.0  #@param {type:"number"}
BATCH_SIZE = 32  #@param {type:"integer"}
DET_BATCH_SIZE = 16  #@param {type:"integer"}
EPOCHS = 40  #@param {type:"integer"}
EXTRA_OPTS = ""  #@param {type:"string"}

PLAN_ARGS = {
    "main_fold0_only": "--only kidneymtl_main --folds 0",
    "tier1_3folds": "--tier1 --max-folds 3",
    "tier1": "--tier1",
    "baselines": "--groups baselines",
    "ablations": "--groups ablations",
    "leakage": "--groups leakage",
    "detection": "--groups detection",
    "all": "",
}[PLAN]
OPTS = (f"experiment.output_root={RUNS_DIR} experiment.work_dir={WORK_DIR} data.num_workers={NUM_WORKERS} "
        f"train.batch_size={BATCH_SIZE} train.det_batch_size={DET_BATCH_SIZE} train.epochs={EPOCHS} {EXTRA_OPTS}")
!python scripts/run_experiments.py {PLAN_ARGS} --dry-run --opts {OPTS}

In [ ]:
#@title 4.2 TRAIN (keep this tab open; re-run after a disconnect to continue)
!python scripts/run_experiments.py {PLAN_ARGS} --hours {HOURS} --opts {OPTS}

In [ ]:
#@title 4.3 Progress (safe to run at any time, also from a new session after 1.1 + 1.3)
import glob
import pandas as pd
done = sorted(glob.glob(f"{RUNS_DIR}/*/fold*/DONE"))
print(f"{len(done)} finished runs")
summ = []
for d in done:
    s = Path(d).parent / "summary.json"
    if s.exists():
        j = json.loads(s.read_text())
        summ.append({k: j.get(k) for k in ("experiment", "fold", "test_accuracy", "test_macro_f1",
                                            "det_mAP@0.5", "ext_stone_sensitivity", "best_epoch")})
if summ:
    display(pd.DataFrame(summ).round(4))
st = Path(RUNS_DIR) / "_status.csv"
if st.exists():
    display(pd.read_csv(st, header=None, names=["time", "experiment", "fold", "exit_code", "minutes"]).tail(15))
errs = glob.glob(f"{RUNS_DIR}/*/fold*/error.log")
for e in errs:
    print("FAILED RUN:", e)
    print(open(e).read()[-1500:])

## 5. Analyse
Builds every table, figure and statistic for the paper from what is on Drive. You can run it on partial results.

In [ ]:
#@title 5.1 Tables, statistics, figures
!python scripts/aggregate_results.py --runs-root "{RUNS_DIR}" --out "{RESULTS_DIR}"
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS_DIR}/RESULTS.md", encoding="utf-8").read()))

In [ ]:
#@title 5.2 Explainability (Grad-CAM + pointing game) and efficiency (needs the GPU)
!python scripts/explain.py --runs-root "{RUNS_DIR}" --out "{RESULTS_DIR}/explain" --experiments kidneymtl_main b_cdr_vgg19 b_resnet50 b_efficientnet_b3 b_convnext_tiny b_vit_small
!python scripts/benchmark_efficiency.py --out "{RESULTS_DIR}/tables"

In [ ]:
#@title 5.3 (optional) YOLOv8 detection reference, then re-aggregate
RUN_YOLO = False  #@param {type:"boolean"}
if RUN_YOLO:
    !pip install -q ultralytics
    !python scripts/train_yolo_baseline.py --model yolov8n.pt --output-root "{RUNS_DIR}"
    !python scripts/aggregate_results.py --runs-root "{RUNS_DIR}" --out "{RESULTS_DIR}"

## 6. Download the results

In [ ]:
#@title 6.1 Zip everything (without model weights) and download it
!python scripts/package_results.py --results-dir "{RESULTS_DIR}" --out "{DRIVE_DIR}/downloads"
zips = sorted(glob.glob(f"{DRIVE_DIR}/downloads/results_*.zip"))
print("Saved on Drive:", zips[-1])
from google.colab import files
files.download(zips[-1])

### Then, on your own computer
1. Put the downloaded `results_YYYYMMDD_HHMM.zip` in the project folder (`Kidney_stone_research/`).
   If the browser download fails, take it from Drive: `MyDrive/Kidney_research/downloads/`.
2. Run:
```
python scripts/import_results.py results_YYYYMMDD_HHMM.zip
python scripts/aggregate_results.py
```
3. `results/RESULTS.md`, `results/tables/*.tex` and `results/figures/` are now local, ready for the next step: writing
   the results and discussion into `paper/main.tex`.

Model weights stay on Drive (`results/runs/<experiment>/fold*/best.pt`). Add `--include-checkpoints` to 6.1 only if
you really need them locally, because they make the zip several GB.